In [ ]:
import torch
import torch.nn.functional as F
import os

# ================= Configuration =================
ENTITY_VEC_FILE = "entity2vec.txt"
RELATION_VEC_FILE = "relation2vec.txt"
DATA_DIR = "./data/"  # Directory containing entity2id.txt and relation2id.txt
P_NORM = 1  # L1 norm (Manhattan distance), usually better for TransE

# ================= Helper Functions =================


def load_id_mappings(in_path):
    """Load entity and relation ID mappings."""
    print("Loading ID mappings...")
    entity2id, id2entity = {}, {}
    relation2id, id2relation = {}, {}

    # Load entities
    with open(os.path.join(in_path, "entity2id.txt"), "r") as f:
        f.readline()  # Skip count
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                name, idx = parts[0], int(parts[1])
                entity2id[name] = idx
                id2entity[idx] = name

    # Load relations
    with open(os.path.join(in_path, "relation2id.txt"), "r") as f:
        f.readline()  # Skip count
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                name, idx = parts[0], int(parts[1])
                relation2id[name] = idx
                id2relation[idx] = name

    return entity2id, id2entity, relation2id, id2relation


def load_vectors(file_path):
    """Load embedding vectors from file into a Tensor."""
    print(f"Loading vectors from {file_path}...")
    vectors = []
    with open(file_path, "r") as f:
        for line in f:
            vec = [float(x) for x in line.strip().split()]
            vectors.append(vec)
    return torch.tensor(vectors)


def find_closest(target_vec, candidate_matrix, k=10, p_norm=1):
    """Find top-k closest vectors using broadcasting."""
    if target_vec.dim() == 1:
        target_vec = target_vec.unsqueeze(0)

    # Calculate distance: || candidates - target ||_p
    dists = torch.norm(candidate_matrix - target_vec, p=p_norm, dim=-1)
    top_val, top_idx = torch.topk(dists, k, largest=False)
    return top_val, top_idx


# ================= Main Execution =================


def main():
    # 1. Load Mappings
    if not os.path.exists(DATA_DIR):
        print(f"Error: Data directory {DATA_DIR} not found.")
        return
    e2id, id2e, r2id, id2r = load_id_mappings(DATA_DIR)

    # 2. Load Vectors
    if not os.path.exists(ENTITY_VEC_FILE) or not os.path.exists(RELATION_VEC_FILE):
        print("Error: Vector files not found.")
        return

    ent_emb = load_vectors(ENTITY_VEC_FILE)
    rel_emb = load_vectors(RELATION_VEC_FILE)

    # Normalize vectors (Standard practice for TransE inference)
    ent_emb = F.normalize(ent_emb, p=2, dim=-1)
    rel_emb = F.normalize(rel_emb, p=2, dim=-1)

    print("\n" + "=" * 50)
    print("Executing Queries")
    print("=" * 50)

    # ---------------------------------------------------------
    # Query 1: Given Head Q30, Relation P36 -> Find Tail
    # Logic: h + r ≈ t
    # ---------------------------------------------------------
    head_name = "Q30"
    rel_name = "P36"

    if head_name in e2id and rel_name in r2id:
        h_idx = e2id[head_name]
        r_idx = r2id[rel_name]

        h_vec = ent_emb[h_idx]
        r_vec = rel_emb[r_idx]

        # Target vector for tail
        target_tail = h_vec + r_vec

        print(f"\n[Query 1] Head: {head_name} + Rel: {rel_name} => Predict Tail")
        vals, idxs = find_closest(target_tail, ent_emb, k=10, p_norm=P_NORM)

        for i in range(10):
            idx = idxs[i].item()
            name = id2e.get(idx, str(idx))
            print(f"  {i + 1}. {name} (ID: {idx}) - Dist: {vals[i].item():.4f}")
    else:
        print(f"\n[Skip] Entity {head_name} or Relation {rel_name} not found.")

    # ---------------------------------------------------------
    # Query 2: Given Head Q30, Tail Q49 -> Find Relation
    # Logic: h + r ≈ t  =>  r ≈ t - h
    # ---------------------------------------------------------
    head_name = "Q30"
    tail_name = "Q49"

    if head_name in e2id and tail_name in e2id:
        h_idx = e2id[head_name]
        t_idx = e2id[tail_name]

        h_vec = ent_emb[h_idx]
        t_vec = ent_emb[t_idx]

        # Target vector for relation
        target_rel = t_vec - h_vec

        print(f"\n[Query 2] Head: {head_name} + Tail: {tail_name} => Predict Relation")
        vals, idxs = find_closest(target_rel, rel_emb, k=10, p_norm=P_NORM)

        for i in range(10):
            idx = idxs[i].item()
            name = id2r.get(idx, str(idx))
            print(f"  {i + 1}. {name} (ID: {idx}) - Dist: {vals[i].item():.4f}")
    else:
        print(f"\n[Skip] Entity {head_name} or {tail_name} not found.")


if __name__ == "__main__":
    main()